# 12 · Gui.Desk 指标 —— `btplot` 一行画进 minibt desk

**模式：desk 界面 + btplot（指标）**
运行：`python tutorials/plots/12_desk_btplot.py`（脚本）/ 本 notebook（单元格）

* `ind.btplot(gui=Gui.Desk)` → desk 打开**本地图表**，把指标「加到图表」；
* `PlotInfo` 样式（`linestyle` 颜色 / 线宽 / 虚线、`overlap` 主副图、
  `legend`）在这条路径上一样生效；
* desk 显示的是 minibt **算好的线**，不会重算，所以信号 / 停止线也保留 ✓。

> **Jupyter**：desk 自动用**浏览器壳 + iframe** 内嵌（不阻塞 kernel）；
> 普通脚本则弹 Qt 桌面窗口，关窗即结束。


In [ ]:
import os
import sys

# 找到仓库根目录（notebook 的 cwd 通常就是 tutorials/plots）→ 优先用仓库里的
# minibt，避免被 site-packages 里的旧副本遮蔽
_here = os.path.abspath(os.getcwd())
while _here and not os.path.isdir(os.path.join(_here, 'minibt')):
    _parent = os.path.dirname(_here)
    if _parent == _here:
        break
    _here = _parent
if os.path.isdir(os.path.join(_here, 'minibt')):
    sys.path.insert(0, _here)

import numpy as np

from minibt import *                      # noqa: F401,F403


In [ ]:
from minibt import BtIndicator, Gui, LineDash, LineStyle, LocalDatas


class MaBand(BtIndicator):
    """一条主图均线 + 一条副图带宽（最简的指标 → desk 图表）。"""

    lines = ('ma', 'band')
    params = dict(length=20, k=2.0)

    #: 均线叠在主图、带宽单独一个副图
    overlap = dict(ma=True, band=False)

    linestyle = dict(
        ma=LineStyle(line_color='#f6c343', line_width=2),
        band=LineStyle(line_color='#2196f3', line_width=1,
                       line_dash=LineDash.dashed),
    )

    #: 副图这条不进左上角图例行（想看名字就删掉这一行）
    legend = dict(band=False)

    def next(self):
        ma = self.close.rolling(int(self.params.length)).mean()
        band = (self.close.rolling(int(self.params.length)).std()
                * float(self.params.k))
        return ma.to_numpy(), band.to_numpy()

In [ ]:
kline = LocalDatas.l2609_60.kline

# ★ 唯一区别：gui=Gui.Desk（等价 gui='desk'）
MaBand(kline).btplot(gui=Gui.Desk)

## 在 desk 里可以继续做什么

* 指标：工具栏「指标」再叠加其它指标、⚙ 就地改参数；
* 布局 / 联动 / 回放 / 画线 / 预警 / 截图：和手动打开的本地图表完全一样；
* 想换引擎只改 `gui`：`Gui.Pylightcharts`（原生窗口）/
  `Gui.PylightchartsWeb`（浏览器标签）/ `gui='jupyter'`（Notebook 内联网页版）。